# UniSpecRec(SGFCF) 网格搜索

这个 Notebook 用于：
- 先使用 SGFCF 生成 CF 预测矩阵
- 再对 UniSpecRec 的 `power` 和 `gamma` 进行网格搜索
- 输出最佳参数，并在验证集和测试集上评估最终结果

In [1]:
import sys
sys.path.append('..')

from utils import DataHandler, Metric, grid_search_unispecrec_hyperparamters
from models import SGFCF, UniSpecRec

In [2]:
BEST_SGFCF_PARAMS = {
    'books': {'k': 2500, 'beta_1': 1.0, 'beta_2': 1.0, 'alpha': 0.0, 'eps': 0.26, 'gamma': 2.5},
    'games': {'k': 130, 'beta_1': 1.0, 'beta_2': 1.2, 'alpha': 19.0, 'eps': 0.5, 'gamma': 0.5},
    'toys': {'k': 440, 'beta_1': 0.5, 'beta_2': 1.7, 'alpha': 9.0, 'eps': 0.5, 'gamma': 2.5},
}


def build_sgfcf_model(datahandler, interaction_data='games', device='cuda', **overrides):
    params = dict(BEST_SGFCF_PARAMS[interaction_data])
    for key, value in overrides.items():
        if value is not None:
            params[key] = value

    model = SGFCF(
        rate_matrix=datahandler.rate_matrix,
        model_config=SGFCF.ModelConfig(
            k=params['k'],
            beta_1=params['beta_1'],
            beta_2=params['beta_2'],
            alpha=params['alpha'],
            eps=params['eps'],
            gamma=params['gamma'],
            device=device,
        ),
    )
    return model, params

In [3]:
# ===== 可修改配置 =====
interaction_data = 'games'     # books / games / toys
semantic_data = 'nvidia'         # nvidia / llama / qwen
normalization_method = 'zscore'  # direct/max/minmax/zscore/l2/softmax
eval_split = 'test'           # valid / test
device = 'cuda'

# 可选：覆盖 SGFCF 参数，保持 None 则使用 BEST_SGFCF_PARAMS
sgfcf_overrides = {
    'k': None,
    'beta_1': None,
    'beta_2': None,
    'alpha': None,
    'eps': None,
    'gamma': None,
}

In [ ]:
datahandler = DataHandler(
    interaction_data=interaction_data,
    semantic_data=semantic_data,
    device=device,
    seed=42,
)
metric = Metric(device=device)

sgfcf_model, sgfcf_params = build_sgfcf_model(
    datahandler=datahandler,
    interaction_data=interaction_data,
    device=device,
    **sgfcf_overrides,
)

# SGFCF 作为 CF 基座
cf_pred_matrix = sgfcf_model.predict()
eval_rate_matrix = datahandler.valid_rate_matrix if eval_split == 'valid' else datahandler.test_rate_matrix

# 直接复用 utils/grid_search.py 中的实现
best_power, best_gamma, best_result = grid_search_unispecrec_hyperparamters(
    cf_pred_matrix=cf_pred_matrix,
    datahandler=datahandler,
    metric=metric,
    normalization_method=normalization_method,
    eval_rate_matrix=eval_rate_matrix,
    show_progress=True,
)

best_model = UniSpecRec(
    cf_pred_matrix=cf_pred_matrix,
    user_semantic_embeddings=datahandler.user_semantic_embeddings,
    item_semantic_embeddings=datahandler.item_semantic_embeddings,
    rate_matrix=datahandler.rate_matrix,
    model_config=UniSpecRec.ModelConfig(
        power=best_power,
        gamma=best_gamma,
        normalization_method=normalization_method,
        device=device,
    ),
)

valid_result = best_model.test(metric=metric, test_rate_matrix=datahandler.valid_rate_matrix)
print('Validation:', ', '.join([f"{k}: {v:.4f}" for k, v in valid_result.items()]))
test_result = best_model.test(metric=metric, test_rate_matrix=datahandler.test_rate_matrix)
print('Test:', ', '.join([f"{k}: {v:.4f}" for k, v in test_result.items()]))
